<a href="https://colab.research.google.com/github/narpavi-ai/cctp-481-notes/blob/main/notebooks/03-tools-data-memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Tools, Data & Memory

**CCTP 481: Building Your First AI Agent · Module 3**

---

### Where we left off

Your food truck agent can now check the weather and check your stock. In Lab 2
you asked it whether to open at Hawrelak, and it gave you a confident
recommendation **based on a threshold it made up.**

It had facts. It did not have **your rules**, and it did not have a **memory**.

### What you'll do here

Fix both.

1. **Memory** — so the agent follows a conversation instead of restarting every time
2. **Your handbook** — so it answers policy questions by *quoting your rule*
   rather than inventing one
3. **Long-term memory** — so something a customer told you on Monday is still
   known on Friday

And then you'll ask the Hawrelak question one more time, and get a different
kind of answer.

⏱️ About 40 minutes.

<p align="center">
<img src="https://raw.githubusercontent.com/narpavi-ai/cctp-481-notes/main/images/lab3.png" alt="The same food truck, wired to weather, your handbook and your stock, with a loop showing memory" width="640">
</p>

### Setup

In [ ]:
%pip install -q -U "langchain[google-genai]>=1.3,<2" "langgraph>=1.2,<2"

print("✅ Installed.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ Key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google AI Studio API key: ")
    print("✅ Key loaded for this session only.")

In [ ]:
from IPython.display import Markdown, display

# One look for everything the model and the tools say, so a student can tell at
# a glance where the notebook stops talking and the model starts.

ANSWER_LIMIT = 1500


def _text_of(message):
    """The readable text of a model message, whatever shape it arrives in.

    Gemini 3.x returns .content as a LIST of blocks, not a string, so the
    obvious str(response.content) prints a Python list with a base64 signature
    inside it. Everything below goes through here.

    Deliberately does NOT touch .text: calling it is deprecated in LangChain 1.x
    and printed a warning above every single answer.
    """
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n\n".join(p for p in parts if p)
    return str(content)


def _card(body, label=None, icon="", limit=ANSWER_LIMIT):
    """A labelled, indented block. Markdown inside still renders."""
    body = _text_of(body).strip()
    if not body:
        body = "*(nothing came back)*"
    if len(body) > limit:
        body = body[:limit].rstrip() + f"\n\n*… trimmed here — {len(body):,} characters in full*"
    lines = ["> " + line for line in body.splitlines()]
    if label:
        lines = [f"> {icon} **{label}**".replace(">  ", "> "), ">"] + lines
    return "\n".join(lines)


def show(response, label="Model answer"):
    """Display a model response as a readable answer card."""
    display(Markdown(_card(response, label, icon="🤖")))


def show_text(text, label=None, icon="📄"):
    """Display plain text - a tool result, a lookup - in the same card."""
    display(Markdown(_card(text, label, icon=icon, limit=2500)))


def show_trace(result, label="Agent trace"):
    """Render every step the agent took, in order, as readable cards."""
    msgs = result["messages"]
    out = [f"#### 🔍 {label} — {len(msgs)} steps", ""]

    for i, m in enumerate(msgs, 1):
        kind = type(m).__name__.replace("Message", "").upper()

        if kind == "HUMAN":
            out += [_card(m, f"{i} · You asked", icon="👤", limit=600), ""]

        elif kind == "TOOL":
            name = getattr(m, "name", "tool")
            out += [_card(m, f"{i} · Tool returned — {name}", icon="🛠️", limit=800), ""]

        elif kind == "AI":
            calls = getattr(m, "tool_calls", None)
            if calls:
                steps = []
                for tc in calls:
                    args = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                    steps.append(f"`{tc['name']}({args})`")
                said = _text_of(m).strip()
                body = "\n\n".join(steps + ([said] if said else []))
                out += [_card(body, f"{i} · Model called a tool", icon="🔧", limit=600), ""]
            else:
                out += [_card(m, f"{i} · Model answered", icon="🤖", limit=900), ""]

        else:
            out += [_card(m, f"{i} · {kind}", limit=600), ""]

    display(Markdown("\n".join(out)))


print("✅ Display helpers ready — use show() instead of print() from here on.")


#### Create the model

Every notebook is its own Colab runtime, so `model` does not carry over from the
last lab — you build it again here. Same one line as Lab 1.

`gemini-3.5-flash-lite` is the pin, and the reason is **quota, not cleverness**:
on the free tier the full Flash models allow **20 requests a day** and these labs
need roughly 70. Check your own limits at <https://aistudio.google.com/rate-limit>.


In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.5-flash-lite"

model = init_chat_model(MODEL)
print(f"✅ Model ready: {MODEL}")


### Your tools from Lab 2

Same two tools, carried forward. Run this and move on — there's nothing new here.

In [ ]:
import requests
from langchain.tools import tool
from langchain.agents import create_agent

WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "freezing fog", 51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
    61: "light rain", 63: "rain", 65: "heavy rain", 66: "freezing rain", 67: "heavy freezing rain",
    71: "light snow", 73: "snow", 75: "heavy snow", 77: "snow grains",
    80: "rain showers", 81: "heavy rain showers", 82: "violent rain showers",
    85: "snow showers", 86: "heavy snow showers",
    95: "thunderstorm", 96: "thunderstorm with hail", 99: "severe thunderstorm with hail",
}
STOCK = {"cinnamon buns": 4, "saskatoon berry pies": 11, "bison chili": 0, "cold brew": 26}


@tool
def get_weather() -> str:
    """Get the CURRENT weather in Edmonton, Alberta.

    Use this for any question about weather, temperature, wind, rain or snow,
    or whether conditions are suitable for opening the food truck.
    Takes no input - it always reports Edmonton.
    """
    now = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": 53.5461, "longitude": -113.4938,
                "current": "temperature_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
                "timezone": "America/Edmonton"},
        timeout=10,
    ).json()["current"]
    return (f"Edmonton weather as of {now['time']}: "
            f"{WEATHER_CODES.get(now['weather_code'], 'unknown conditions')}, "
            f"{now['temperature_2m']}°C (feels like {now['apparent_temperature']}°C), "
            f"wind {now['wind_speed_10m']} km/h, precipitation {now['precipitation']} mm.")


@tool
def check_stock(item: str) -> str:
    """Look up how many units of a menu item are currently in the truck.

    Use this for questions about inventory, stock levels, or availability.
    """
    count = STOCK.get(item.lower().strip())
    if count is None:
        return f"No menu item called {item!r}. The menu is: {list(STOCK)}"
    return f"{item}: {count} in the truck"


print("✅ Tools ready:", [get_weather.name, check_stock.name])

### Step 1 — See the memory problem clearly

Ask a question, then ask a **follow-up** that only makes sense if it remembers
the first one.

In [ ]:
agent = create_agent(
    model=model, tools=[check_stock],
    system_prompt="You are an assistant for an Edmonton food truck. Use your tools.",
)

a = agent.invoke({"messages": [{"role": "user", "content": "How many cinnamon buns do I have?"}]})
show(a["messages"][-1], "Q1 — How many cinnamon buns do I have?")

b = agent.invoke({"messages": [{"role": "user", "content": "And how about the cold brew?"}]})
show(b["messages"][-1], "Q2 — And how about the cold brew?")

Depending on the day it either guessed what you meant or asked you to repeat
yourself — but either way it **had no idea Q1 ever happened.** Each `invoke()`
was a brand new conversation, exactly as in Lab 1.

### Step 2 — Short-term memory: a checkpointer

One extra argument. Watch for it.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[check_stock],
    system_prompt="You are an assistant for an Edmonton food truck. Use your tools.",
    checkpointer=InMemorySaver(),      # ← the only new line
)

config = {"configurable": {"thread_id": "monday-morning"}}

a = agent.invoke({"messages": [{"role": "user", "content": "How many cinnamon buns do I have?"}]}, config=config)
show(a["messages"][-1], "Q1")

b = agent.invoke({"messages": [{"role": "user", "content": "And how about the cold brew?"}]}, config=config)
show(b["messages"][-1], "Q2 — with a checkpointer")

**🎯 Checkpoint.** It followed the thread. It understood that "and how about the
cold brew" meant *"how many cold brews are in the truck."*

Two things made that work, and you need both:

| | What it does |
|---|---|
| `checkpointer=InMemorySaver()` | **Where** the conversation gets saved |
| `thread_id` in the config | **Which** conversation this message belongs to |

Remember Lab 1, where you re-sent the whole message list by hand? The
checkpointer is doing exactly that, for you, on every call.

### Step 3 — One thread, one customer

`thread_id` isn't decoration. It's the wall between conversations. Ask the same
follow-up on a **different** thread:

In [ ]:
other = {"configurable": {"thread_id": "someone-else-entirely"}}
r = agent.invoke({"messages": [{"role": "user", "content": "And how about the cold brew?"}]}, config=other)
show(r["messages"][-1], "Same question, different thread_id")

**💡 This is a design decision, not a detail.** If you serve two customers and
give them the same `thread_id`, they read each other's conversation. One
customer per thread. Get this wrong in production and it's a privacy incident,
not a bug.

### Step 4 — Give it your handbook

Here is the thing your agent has been missing since Lab 2: **your actual rules.**

Real businesses run on documents — a staff handbook, an SOP, a policy binder.
None of it was ever in the model's training data, and it has no way to look.
So you hand it over as a tool.

**Read the first policy carefully.** It's the one that answers the question you
started this course with.

In [ ]:
POLICIES = [
    "Cold and storms: we do not open when the temperature feels colder than -20C, "
    "or when Environment Canada has a thunderstorm warning in effect. "
    "Propane and high wind do not mix.",

    "Refunds: customers may return any item within 24 hours of purchase for a full "
    "refund, no receipt required.",

    "Allergens: cinnamon buns and saskatoon berry pies are made in a kitchen that "
    "handles nuts, dairy, wheat and eggs. We cannot guarantee any item is nut-free.",

    "Locations: Hawrelak Park on weekends, Louise McKinney for downtown events, "
    "Old Strathcona Farmers' Market on Saturday mornings.",

    "Minimum stock: we do not open a service with fewer than 6 cinnamon buns, "
    "because they are what most people queue for.",

    "Propane: propane tanks are checked every morning before service and swapped "
    "when below one quarter. Never run a service on a tank below one quarter.",
]

print(f"✅ {len(POLICIES)} policies loaded")

In [ ]:
@tool
def search_policies(query: str) -> str:
    """Search the food truck's staff handbook for official policy.

    Use this for any question about rules, refunds, allergens, opening or
    closing, weather closures, minimum stock, locations, or propane safety.
    Pass the key words from the question as the query.
    """
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [p for p in POLICIES if any(w in p.lower() for w in words)]
    if not hits:
        return "No matching policy found. Tell the user you will check with the owner."
    return "\n".join(f"- {h}" for h in hits)


show_text(search_policies.invoke({"query": "cold weather closure"}),
          "Searching the handbook for: cold weather closure")

> **Being honest about what this is.** That's keyword matching — it finds
> policies containing words from the question. Real systems use **embeddings**,
> which match on *meaning*, so "is it too chilly to trade?" would still find the
> cold-weather rule. Same shape of tool, better matching underneath. You'll build
> that in **CCTP 482**; here the mechanism stays visible on purpose.

### Step 5 — Ask the Hawrelak question again

This is the moment the whole course has been building to. Same agent, same
weather, same stock — plus the handbook.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_weather, check_stock, search_policies],
    system_prompt=(
        "You are an assistant for a food truck in Edmonton's river valley. "
        "ALWAYS search the handbook before making a recommendation about opening, "
        "refunds, allergens or safety, and quote the policy you relied on. "
        "Never invent a policy. If no policy matches, say so and say you will "
        "check with the owner."
    ),
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "should-i-open"}}
r = agent.invoke({"messages": [{"role": "user", "content":
    "Should I open at Hawrelak Park today? Tell me exactly what you based that on."}]}, config=cfg)

show(r["messages"][-1], "Lab 3's answer")

### 💡 Compare that to Lab 2

Same question. Same tools for weather and stock. The difference is that it now
**cites a rule that exists**, instead of a threshold it invented.

| | Lab 2 | Lab 3 |
|---|---|---|
| Live weather | ✅ | ✅ |
| Stock count | ✅ | ✅ |
| A threshold | 🚫 made up | ✅ *"feels colder than −20 °C"* — your rule |
| A minimum stock rule | 🚫 didn't exist | ✅ *"fewer than 6 cinnamon buns"* |
| If you disagree with it | you can't tell why it said that | you can go edit the policy |

That last row is the real prize. **The behaviour is now in a document you own,
not buried in a model you don't.** Change the handbook, and the agent's advice
changes — no code, no retraining.

### Step 5b — Force the interesting case

Today's real Edmonton weather might be pleasant, in which case your agent
sensibly said "open." Give it the cold scenario explicitly so you can see the
rule actually fire.

In [ ]:
cold = {"configurable": {"thread_id": "cold-scenario"}}
r = agent.invoke({"messages": [{"role": "user", "content":
    "Suppose tomorrow morning it feels like -24C at Hawrelak with a thunderstorm "
    "warning in effect, and I have 3 cinnamon buns. Should I open? Quote the policy."}]},
    config=cold)

show(r["messages"][-1], "The cold scenario")

### Step 6 — What happens when there's no rule

The most valuable thing a policy tool can do is **fail loudly**. Ask about
something genuinely not in the handbook.

In [ ]:
r = agent.invoke({"messages": [{"role": "user", "content":
    "What's our policy on dogs at the service window?"}]}, config=cfg)
show(r["messages"][-1], "A question the handbook does not answer")

**💡 The system prompt did real work there.** *"Never invent a policy. If no
policy matches, say so."*

Without that instruction the model will cheerfully write you a dog policy, in
confident handbook prose, that no one ever approved. **A retrieval tool without
a don't-invent instruction is a more convincing liar than no tool at all** —
because now the answer *sounds* sourced.

### Step 7 — Long-term memory

The checkpointer remembers **one conversation**. But some facts need to outlive
the conversation entirely — an allergy, a standing order, an accessibility need.

That's a different problem, and it needs a different tool: one that **writes**.

In [ ]:
CUSTOMER_NOTES = {}


@tool
def remember_about_customer(customer: str, fact: str) -> str:
    """Save a durable fact about a regular customer, such as an allergy,
    an accessibility need, or a standing order.

    Use this when a customer states something worth recalling on a future visit.
    """
    CUSTOMER_NOTES.setdefault(customer.lower(), []).append(fact)
    return f"Saved for {customer}: {fact}"


@tool
def recall_about_customer(customer: str) -> str:
    """Retrieve everything previously saved about a named customer.

    Use this before advising on an order for someone the truck knows.
    """
    notes = CUSTOMER_NOTES.get(customer.lower())
    return "; ".join(notes) if notes else f"Nothing on file for {customer}."


agent = create_agent(
    model=model,
    tools=[get_weather, check_stock, search_policies,
           remember_about_customer, recall_about_customer],
    system_prompt=(
        "You are an assistant for an Edmonton food truck. Save durable customer "
        "facts when you learn them, and check what you already know about a "
        "customer before advising on their order. Always search the handbook "
        "before answering a policy question, and never invent a policy."
    ),
    checkpointer=InMemorySaver(),
)

monday = {"configurable": {"thread_id": "monday-market"}}
r = agent.invoke({"messages": [{"role": "user", "content":
    "Sam just told me she's severely allergic to nuts."}]}, config=monday)
show(r["messages"][-1], "Monday")

In [ ]:
# A different day. A different conversation. A different thread_id.
friday = {"configurable": {"thread_id": "friday-hawrelak"}}
r = agent.invoke({"messages": [{"role": "user", "content":
    "Sam's at the window and wants a cinnamon bun. Anything I should know?"}]}, config=friday)
show(r["messages"][-1], "Friday")

**🎯 Checkpoint.** Different conversation, different thread — and it still knew.

Look at what it had to combine to answer that well: the **saved fact** (Sam's
allergy) and the **handbook** (cinnamon buns are made in a kitchen that handles
nuts). Neither alone gets you to the right answer.

### The distinction worth carrying out of this course

| | Short-term memory | Long-term memory |
|---|---|---|
| What it is | the checkpointer | a tool that writes to storage |
| Scope | one `thread_id` | every conversation, forever |
| Who decides what's kept | automatic — everything | **the agent does**, by calling the tool |
| Here | `InMemorySaver()` | `CUSTOMER_NOTES` |
| In production | Postgres, Redis | a real database, and a retention policy |

**⚠️ A governance point that is not hypothetical.** You just built a system that
permanently records health information about a named person, on the strength of
a model's judgement that it was worth saving. Nobody consented to that. Nobody
can see the file. Nobody can ask for it to be deleted.

`InMemorySaver` forgets when the runtime stops, which is why this is safe in a
classroom. **A real deployment is a records system**, and every question you'd
ask about a filing cabinet — who can read it, how long it's kept, how someone
gets themselves removed — you now have to answer about your agent. Module 4
is where that stops being a footnote.

### Your turn

1. **Write your own handbook.** Replace `MY_POLICIES` below with three real rules
   from your own work. Point `search_policies` at it, rebuild the agent, and ask
   it something a new employee would ask.
2. **Then ask it something your handbook doesn't cover** and confirm it says so
   instead of inventing. If it invents, strengthen the system prompt until it stops.
3. **Break the wall.** Give two different "customers" the same `thread_id` and
   watch one see the other's conversation. It takes ten seconds and you'll never
   forget it.

In [ ]:
MY_POLICIES = [
    "...",
    "...",
    "...",
]

# @tool
# def search_my_policies(query: str) -> str:
#     """Search <your organisation>'s handbook for official policy. Use this for..."""
#     ...

## If something breaks

| What you see | What it means | Fix |
|---|---|---|
| `404 NOT_FOUND` | Google retired that model | Check <https://aistudio.google.com> and edit `MODEL` |
| It still forgets between calls | Missing `checkpointer=`, or you changed `thread_id` | Both are required, and the `thread_id` must match |
| `ValueError` about a missing checkpointer | You passed a `config` with a `thread_id` but built the agent without one | Add `checkpointer=InMemorySaver()` |
| It invents a policy anyway | The system prompt isn't firm enough | Say *never invent a policy* explicitly, and tell it what to do instead |
| `search_policies` finds nothing obvious | Keyword matching, not meaning | Use words that appear in the policy text — or note it, which is exactly why embeddings exist |
| Sam isn't remembered on Friday | The model didn't call the save tool on Monday | Read Monday's trace. If there's no `🔧 calls remember_about_customer`, the docstring or system prompt didn't push hard enough |

## What you learned

- `checkpointer` + `thread_id` give an agent **short-term memory**; one thread per customer
- Your documents reach the agent as **a tool**, not as training
- A retrieval tool needs an explicit **never-invent** instruction, or it makes things worse
- **Long-term memory is a tool that writes** — and the agent decides what's worth keeping
- Behaviour that lives in a handbook is behaviour **you can edit**

## References

**LangChain docs**

- [Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)
- [Long-term memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)
- [Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval) — and what embeddings add
- [Tools](https://docs.langchain.com/oss/python/langchain/tools)

**Beyond LangChain**

- [Anthropic — Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)

### The no-code version of this lab

**[`n8n/03-tools-data-memory.json`](n8n/03-tools-data-memory.json)** — the Simple
Memory sub-node is the checkpointer, and Session ID is the `thread_id`.
See **[`n8n/SETUP.md`](n8n/SETUP.md)**.

### Next

**Lab 4 — Human Oversight & Evaluation.** Your agent can now look things up,
quote your rules and remember your customers. It can also *act* — and in Lab 2 it
placed an order without asking anybody. Next you put a human in front of the
expensive decisions, build a rule the model cannot argue past, and find out
whether your agent is actually any good.